# Hybrid Multimodal Transformer Framework for Brain Tumor Detection
**Based on the uploaded methodology:** CT + MRI + clinical data; preprocessing; NeuroMamba-3D-inspired volumetric encoder; ClinEmbed; NeuroAlign; TriFuse-Former; uncertainty-guided refinement (UGR); missing-modality experts; X-Neuro-style explanations; patient-level validation.

> **Important:** The document defines the proposed methodology but does not provide a dataset, pretrained NeuroMamba/Mamba-2 weights, or exact model hyperparameters. This notebook is a runnable research prototype, not a clinically validated system. It uses a compact 3D CNN and a state-space-inspired sequence block as practical stand-ins. Replace the synthetic demo data with a properly consented, patient-level dataset before reporting scientific results. Never use the demo metrics as evidence of clinical performance.

## 1. Install dependencies
Run this cell in Google Colab or a Python environment. NIfTI support is optional; `.npy` volumes work without it.

In [ ]:
%pip -q install numpy pandas scikit-learn scipy matplotlib seaborn tqdm openpyxl

## 2. Imports, reproducibility, and configuration

In [ ]:
import os, json, random, warnings, zipfile, math, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             balanced_accuracy_score)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Set this to True to create a small synthetic dataset and verify the whole pipeline.
# Set False and configure MANIFEST_PATH for your real dataset.
USE_SYNTHETIC_DEMO = True
MANIFEST_PATH = "/content/drive/MyDrive/brain_tumor_manifest.csv"
VOLUME_SHAPE = (32, 64, 64)   # downsampled demonstration shape (D,H,W)
BATCH_SIZE = 4
EPOCHS = 5                   # increase for real experiments
LEARNING_RATE = 2e-4
N_SPLITS = 3                 # increase to 5 for formal validation

## 3. Dataset format and data loading
For real data, create a CSV manifest with columns:
- `patient_id`: unique patient identifier
- `ct_path`: path to CT volume (`.npy` or NIfTI `.nii`/`.nii.gz`)
- `mri_path`: path to MRI volume (`.npy` or NIfTI `.nii`/`.nii.gz`)
- `label`: class label (binary 0/1 or categorical)
- optional clinical columns such as `age`, `sex`, `symptom_score`, `tumor_size`

The split is performed at patient level. If each patient has multiple studies/slices, ensure all records from one patient stay in the same fold. This baseline assumes one manifest row per patient.

In [ ]:
def make_synthetic_data(n=90):
    rng = np.random.default_rng(SEED)
    rows = []
    for i in range(n):
        label = int(i % 2)
        # Weak class-associated synthetic signal for pipeline smoke-testing only.
        ct = rng.normal(0, 1, VOLUME_SHAPE).astype(np.float32)
        mri = rng.normal(0, 1, VOLUME_SHAPE).astype(np.float32)
        if label:
            d0, h0, w0 = 12, 26, 26
            ct[d0-4:d0+4, h0-8:h0+8, w0-8:w0+8] += 1.0
            mri[d0-5:d0+5, h0-7:h0+7, w0-7:w0+7] += 1.2
        rows.append({
            "patient_id": f"SYN_{i:04d}", "ct_volume": ct, "mri_volume": mri,
            "label": label, "age": float(rng.normal(58 + 4*label, 10)),
            "sex": "F" if rng.random() > .5 else "M",
            "symptom_score": float(max(0, rng.normal(2 + 2*label, 1))),
            "tumor_size": float(max(0, rng.normal(1 + 2*label, .7)))
        })
    return pd.DataFrame(rows)

if USE_SYNTHETIC_DEMO:
    df = make_synthetic_data(90)
    print("Using synthetic demo data:", df.shape)
else:
    if not os.path.exists(MANIFEST_PATH):
        raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")
    df = pd.read_csv(MANIFEST_PATH)
    required = {"patient_id", "ct_path", "mri_path", "label"}
    missing = required - set(df.columns)
    if missing: raise ValueError(f"Manifest missing required columns: {missing}")
    if df["patient_id"].duplicated().any():
        raise ValueError("Expected one row per patient. Resolve repeated studies before patient-level CV.")
    df["ct_volume"] = df["ct_path"].apply(load_volume)
    df["mri_volume"] = df["mri_path"].apply(load_volume)

print(df.head(2)[["patient_id", "label"]])
print("Labels:", df["label"].value_counts().to_dict())

## 4. Clinical feature preprocessing
The transformer is fitted on each training fold only, avoiding leakage from validation patients. Numeric columns are imputed and standardized; categorical columns are imputed and one-hot encoded.

In [ ]:
def build_clinical_preprocessor(frame, exclude=("patient_id","ct_path","mri_path","ct_volume","mri_volume","label")):
    cols = [c for c in frame.columns if c not in exclude]
    numeric = [c for c in cols if pd.api.types.is_numeric_dtype(frame[c])]
    categorical = [c for c in cols if c not in numeric]
    try:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    transformers = []
    if numeric:
        transformers.append(("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                                              ("scale", StandardScaler())]), numeric))
    if categorical:
        transformers.append(("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                              ("onehot", cat_encoder)]), categorical))
    if not transformers:
        raise ValueError("No clinical columns found. Add clinical features to the manifest.")
    return ColumnTransformer(transformers, remainder="drop"), cols

def volume_for_row(row, key):
    val = row[key]
    if isinstance(val, np.ndarray): return val.astype(np.float32)
    return load_volume(val)

labels_raw = df["label"].astype(str)
classes = sorted(labels_raw.unique())
label_map = {v:i for i,v in enumerate(classes)}
y = labels_raw.map(label_map).to_numpy(dtype=np.int64)
NUM_CLASSES = len(classes)
print("Label mapping:", label_map, "| classes:", NUM_CLASSES)
if NUM_CLASSES < 2: raise ValueError("At least two label classes are required.")

## 5. Model blocks: NeuroMamba-3D-inspired encoder, ClinEmbed, NeuroAlign, TriFuse-Former, UGR, MoE
The provided methodology names custom modules but does not include source code or pretrained weights. The implementation below operationalizes the concepts with a compact 3D CNN, gated state-space-inspired token mixer, bidirectional cross-attention, gated tri-modal fusion, evidence/uncertainty heads, modality dropout, and mixture-of-experts classification.

In [ ]:
class StateSpaceInspiredMixer(nn.Module):
    """Lightweight sequence mixer (not an official Mamba-2 implementation)."""
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.in_proj = nn.Linear(dim, dim * 2)
        self.depthwise = nn.Conv1d(dim, dim, kernel_size=3, padding=1, groups=dim)
        self.out_proj = nn.Linear(dim, dim)
    def forward(self, x):
        z = self.norm(x)
        a, gate = self.in_proj(z).chunk(2, dim=-1)
        a = self.depthwise(a.transpose(1,2)).transpose(1,2)
        return x + self.out_proj(a * torch.sigmoid(gate))

class NeuroMamba3D(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv3d(1,16,3,padding=1), nn.BatchNorm3d(16), nn.GELU(), nn.MaxPool3d(2),
            nn.Conv3d(16,32,3,padding=1), nn.BatchNorm3d(32), nn.GELU(), nn.MaxPool3d(2),
            nn.Conv3d(32,64,3,padding=1), nn.BatchNorm3d(64), nn.GELU(),
            nn.AdaptiveAvgPool3d((2,2,2)))
        self.project = nn.Linear(64, dim)
        self.mixer = StateSpaceInspiredMixer(dim)
        self.pool = nn.AdaptiveAvgPool1d(1)
    def forward(self, volume):
        fmap = self.cnn(volume)                       # B,C,2,2,2
        tokens = fmap.flatten(2).transpose(1,2)      # B,8,C
        tokens = self.mixer(self.project(tokens))
        pooled = tokens.mean(dim=1)
        return pooled, fmap

class ClinEmbed(nn.Module):
    def __init__(self, clinical_dim, dim=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(clinical_dim, dim), nn.LayerNorm(dim),
                                 nn.GELU(), nn.Dropout(.1), nn.Linear(dim, dim))
    def forward(self, x): return self.net(x).unsqueeze(1)

class NeuroAlign(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        self.ct_to_mri = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mri_to_ct = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.img_to_clin = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
    def forward(self, ct, mri, clin):
        c2m,_ = self.ct_to_mri(ct, mri, mri)
        m2c,_ = self.mri_to_ct(mri, ct, ct)
        im = self.norm((ct + c2m + mri + m2c)/4)
        i2cl,_ = self.img_to_clin(im, clin, clin)
        aligned = self.norm(im + i2cl)
        return aligned, clin

class TriFuseFormer(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        self.cross = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.gates = nn.Sequential(nn.Linear(dim*3, dim*3), nn.Sigmoid())
        self.norm = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Dropout(.1), nn.Linear(dim*2, dim))
    def forward(self, ct, mri, clin):
        tokens = torch.cat([ct,mri,clin], dim=1)
        cross,_ = self.cross(tokens, tokens, tokens)
        pooled = cross.mean(dim=1)
        gate = self.gates(torch.cat([ct.mean(1),mri.mean(1),clin.mean(1)],dim=-1))
        gc,gm,gl = gate.chunk(3,dim=-1)
        weighted = gc*ct.mean(1) + gm*mri.mean(1) + gl*clin.mean(1)
        fused = self.norm(pooled + weighted)
        return self.norm(fused + self.ffn(fused))

class UncertaintyGuidedRefinement(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.uncertainty = nn.Sequential(nn.Linear(dim, dim//2), nn.GELU(), nn.Linear(dim//2, 1), nn.Softplus())
        self.refine = nn.Sequential(nn.Linear(dim, dim), nn.Sigmoid())
    def forward(self, x):
        u = self.uncertainty(x)                 # positive uncertainty proxy
        reliability = 1.0 / (1.0 + u)
        refined = x * self.refine(x) * reliability
        return refined, u.squeeze(-1), reliability.squeeze(-1)

class MultimodalBrainTumorNet(nn.Module):
    def __init__(self, clinical_dim, num_classes, dim=128, modality_dropout=.25):
        super().__init__()
        self.ct_encoder = NeuroMamba3D(dim)
        self.mri_encoder = NeuroMamba3D(dim)
        self.clin_encoder = ClinEmbed(clinical_dim, dim)
        self.align = NeuroAlign(dim)
        self.fusion = TriFuseFormer(dim)
        self.ugr = UncertaintyGuidedRefinement(dim)
        self.modality_dropout = modality_dropout
        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(dim,dim//2),nn.GELU(),nn.Dropout(.1),
                                                    nn.Linear(dim//2,num_classes)) for _ in range(3)])
        self.router = nn.Sequential(nn.Linear(dim,3), nn.Softmax(dim=-1))
        self.evidence = nn.Sequential(nn.Linear(dim,num_classes), nn.Softplus())
    def forward(self, ct, mri, clinical, modality_mask=None):
        ct_vec, ct_map = self.ct_encoder(ct)
        mri_vec, mri_map = self.mri_encoder(mri)
        clin_tok = self.clin_encoder(clinical)
        ct_tok, mri_tok = ct_vec.unsqueeze(1), mri_vec.unsqueeze(1)
        if modality_mask is not None:
            ct_tok = ct_tok * modality_mask[:,0,None,None]
            mri_tok = mri_tok * modality_mask[:,1,None,None]
            clin_tok = clin_tok * modality_mask[:,2,None,None]
        elif self.training and self.modality_dropout > 0:
            mask = (torch.rand(ct.shape[0],3,device=ct.device) > self.modality_dropout).float()
            # Keep at least one modality per patient.
            none = mask.sum(1) == 0
            mask[none, torch.randint(0,3,(int(none.sum()),),device=ct.device)] = 1
            ct_tok = ct_tok * mask[:,0,None,None]
            mri_tok = mri_tok * mask[:,1,None,None]
            clin_tok = clin_tok * mask[:,2,None,None]
        aligned_img, aligned_clin = self.align(ct_tok,mri_tok,clin_tok)
        fused = self.fusion(aligned_img, mri_tok, aligned_clin)
        refined, uncertainty, reliability = self.ugr(fused)
        expert_logits = torch.stack([expert(refined) for expert in self.experts], dim=1)
        route = self.router(refined)
        logits = (expert_logits * route.unsqueeze(-1)).sum(dim=1)
        evidence = self.evidence(refined)
        return {"logits":logits, "uncertainty":uncertainty, "reliability":reliability,
                "route":route, "evidence":evidence, "ct_map":ct_map, "mri_map":mri_map,
                "embedding":refined}

## 6. PyTorch dataset and fold-specific preprocessing

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, frame, clinical_array, y_array):
        self.frame = frame.reset_index(drop=True)
        self.clinical = np.asarray(clinical_array, dtype=np.float32)
        self.y = np.asarray(y_array, dtype=np.int64)
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        ct = volume_for_row(row, "ct_volume")
        mri = volume_for_row(row, "mri_volume")
        return (torch.from_numpy(ct[None]).float(),
                torch.from_numpy(mri[None]).float(),
                torch.from_numpy(self.clinical[idx]).float(),
                torch.tensor(self.y[idx]).long())

def fit_clinical(train_frame, valid_frame):
    prep, _ = build_clinical_preprocessor(train_frame)
    xtr = prep.fit_transform(train_frame)
    xva = prep.transform(valid_frame)
    return np.asarray(xtr,dtype=np.float32), np.asarray(xva,dtype=np.float32), prep

def calculate_metrics(y_true, probs):
    pred = probs.argmax(axis=1)
    out = {
        "accuracy": accuracy_score(y_true,pred),
        "balanced_accuracy": balanced_accuracy_score(y_true,pred),
        "precision_macro": precision_score(y_true,pred,average="macro",zero_division=0),
        "recall_macro": recall_score(y_true,pred,average="macro",zero_division=0),
        "f1_macro": f1_score(y_true,pred,average="macro",zero_division=0)
    }
    try:
        out["roc_auc_ovr_macro"] = roc_auc_score(y_true,probs,multi_class="ovr",average="macro")
    except Exception:
        out["roc_auc_ovr_macro"] = np.nan
    return out

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, ys, ps, unc = 0.0, [], [], []
    for ct,mri,clin,yb in loader:
        ct,mri,clin,yb = ct.to(DEVICE),mri.to(DEVICE),clin.to(DEVICE),yb.to(DEVICE)
        with torch.set_grad_enabled(training):
            out = model(ct,mri,clin)
            ce = F.cross_entropy(out["logits"], yb)
            # Small penalty to discourage extreme uncertainty on correctly predicted cases.
            probs = out["logits"].softmax(-1)
            correct_conf = probs.gather(1,yb[:,None]).squeeze(1)
            loss = ce + 0.01 * (out["uncertainty"] * correct_conf.detach()).mean()
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                optimizer.step()
        total_loss += loss.item()*len(yb)
        ys.extend(yb.detach().cpu().numpy().tolist())
        ps.extend(out["logits"].softmax(-1).detach().cpu().numpy())
        unc.extend(out["uncertainty"].detach().cpu().numpy().tolist())
    return total_loss/len(loader.dataset), np.array(ys), np.array(ps), np.array(unc)

## 7. Patient-level stratified cross-validation
This is a compact demonstration. For publication-grade evaluation, use nested CV or an untouched patient-level test set, tune hyperparameters only inside training folds, and report confidence intervals. A small dataset may not support reliable estimates.

In [ ]:
unique_labels, counts = np.unique(y, return_counts=True)
n_splits = min(N_SPLITS, int(counts.min()))
if n_splits < 2: raise ValueError("Not enough patients per class for stratified cross-validation.")
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
fold_metrics, fold_predictions, fold_uncertainties = [], [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
    print(f"Fold {fold}/{n_splits}")
    tr_df, va_df = df.iloc[tr_idx].copy(), df.iloc[va_idx].copy()
    Xtr, Xva, clinical_prep = fit_clinical(tr_df, va_df)
    train_ds = BrainTumorDataset(tr_df, Xtr, y[tr_idx])
    valid_ds = BrainTumorDataset(va_df, Xva, y[va_idx])
    train_loader = DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
    valid_loader = DataLoader(valid_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    model = MultimodalBrainTumorNet(Xtr.shape[1], NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
    for epoch in range(EPOCHS):
        train_loss, *_ = run_epoch(model,train_loader,optimizer)
    val_loss, yv, probs, unc = run_epoch(model,valid_loader,None)
    metrics = calculate_metrics(yv,probs)
    metrics.update({"fold":fold,"val_loss":val_loss})
    fold_metrics.append(metrics)
    fold_predictions.append((va_idx,yv,probs))
    fold_uncertainties.append((va_idx,unc))
    print({k:round(v,4) if isinstance(v,(float,np.floating)) else v for k,v in metrics.items()})

metrics_df = pd.DataFrame(fold_metrics)
display(metrics_df)
print("\nFold mean ± standard deviation:")
display(metrics_df.drop(columns="fold").agg(["mean","std"]).T)

## 8. Confusion matrix and performance plots

In [ ]:
# Combine out-of-fold predictions
oof_probs = np.zeros((len(y),NUM_CLASSES),dtype=float)
oof_unc = np.zeros(len(y),dtype=float)
for idx, yv, probs in fold_predictions: oof_probs[idx] = probs
for idx, unc in fold_uncertainties: oof_unc[idx] = unc
oof_pred = oof_probs.argmax(1)

cm = confusion_matrix(y,oof_pred,labels=np.arange(NUM_CLASSES))
fig,ax = plt.subplots(figsize=(7,6))
im=ax.imshow(cm,cmap="Blues")
ax.set_title("Out-of-fold confusion matrix",fontweight="bold")
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_xticks(range(NUM_CLASSES),classes,rotation=30); ax.set_yticks(range(NUM_CLASSES),classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]): ax.text(j,i,str(cm[i,j]),ha="center",va="center",fontweight="bold")
fig.colorbar(im,ax=ax); plt.tight_layout(); plt.show()

plot_cols = ["accuracy","precision_macro","recall_macro","f1_macro","roc_auc_ovr_macro"]
available = [c for c in plot_cols if c in metrics_df]
fig,ax=plt.subplots(figsize=(10,5))
means=metrics_df[available].mean()
stds=metrics_df[available].std().fillna(0)
ax.bar(available,means,yerr=stds,capsize=4)
ax.set_ylim(0,1.05); ax.set_ylabel("Score"); ax.set_title("Cross-validation metrics (mean ± SD)",fontweight="bold")
ax.tick_params(axis="x",rotation=25); ax.grid(axis="y",alpha=.25); plt.tight_layout(); plt.show()

print(classification_report(y,oof_pred,labels=np.arange(NUM_CLASSES),target_names=[str(c) for c in classes],zero_division=0))

## 9. Missing-modality robustness and modality ablation
The model accepts a modality mask in order `[CT, MRI, clinical]`, with 1=available and 0=missing. This evaluates the final fold model only as an illustrative smoke test; rigorous ablation should retrain each configuration within each fold.

In [ ]:
# Refit a model on all data for demo inference/ablation. This is not an independent test result.
Xall, _, clinical_prep_all = fit_clinical(df, df)
all_ds = BrainTumorDataset(df, Xall, y)
all_loader = DataLoader(all_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
demo_model = MultimodalBrainTumorNet(Xall.shape[1],NUM_CLASSES).to(DEVICE)
opt = torch.optim.AdamW(demo_model.parameters(),lr=LEARNING_RATE,weight_decay=1e-4)
for ep in range(max(2,EPOCHS)):
    run_epoch(demo_model,all_loader,opt)

def evaluate_mask(model, frame, clinical, labels, mask_tuple):
    ds=BrainTumorDataset(frame,clinical,labels)
    loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False)
    model.eval(); probs_all=[]; y_all=[]
    mask=torch.tensor(mask_tuple,dtype=torch.float32,device=DEVICE).view(1,3)
    with torch.no_grad():
        for ct,mri,clin,yb in loader:
            out=model(ct.to(DEVICE),mri.to(DEVICE),clin.to(DEVICE),
                      modality_mask=mask.expand(len(yb),-1))
            probs_all.extend(out["logits"].softmax(-1).cpu().numpy()); y_all.extend(yb.numpy())
    return calculate_metrics(np.array(y_all),np.array(probs_all))

configs={
    "CT only":(1,0,0),"MRI only":(0,1,0),"Clinical only":(0,0,1),
    "CT + MRI":(1,1,0),"CT + Clinical":(1,0,1),"MRI + Clinical":(0,1,1),
    "All modalities":(1,1,1)
}
ablation=[]
for name,mask in configs.items():
    row={"configuration":name,**evaluate_mask(demo_model,df,Xall,y,mask)}
    ablation.append(row)
ablation_df=pd.DataFrame(ablation)
display(ablation_df)

## 10. X-Neuro-style explanations: uncertainty, clinical feature importance, and 3D saliency
These are lightweight interpretability approximations. Attention weights and saliency are not proof of causal reasoning. SHAP and counterfactuals can be added for a chosen fitted model; for high-dimensional imaging, they can be computationally expensive.

In [ ]:
# Uncertainty distribution
fig,ax=plt.subplots(figsize=(8,4))
ax.hist(oof_unc,bins=20,edgecolor="black")
ax.set_title("Out-of-fold uncertainty proxy",fontweight="bold"); ax.set_xlabel("Predicted uncertainty"); ax.set_ylabel("Patients")
plt.tight_layout(); plt.show()

# Clinical permutation importance on out-of-fold probability loss (simple model-agnostic approximation)
# For this demo, use the fitted clinical preprocessing and the trained demo model; this is descriptive only.
def predict_with_clinical(model, frame, clinical_arr):
    ds=BrainTumorDataset(frame,clinical_arr,np.zeros(len(frame),dtype=int))
    loader=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=False)
    out_probs=[]
    model.eval()
    with torch.no_grad():
        for ct,mri,clin,_ in loader:
            out=model(ct.to(DEVICE),mri.to(DEVICE),clin.to(DEVICE))
            out_probs.extend(out["logits"].softmax(-1).cpu().numpy())
    return np.asarray(out_probs)

base_probs=predict_with_clinical(demo_model,df,Xall)
base_score=accuracy_score(y,base_probs.argmax(1))
rng=np.random.default_rng(SEED)
perm_rows=[]
for j in range(Xall.shape[1]):
    xp=Xall.copy(); xp[:,j]=rng.permutation(xp[:,j])
    pp=predict_with_clinical(demo_model,df,xp)
    perm_rows.append((j,base_score-accuracy_score(y,pp.argmax(1))))
perm_df=pd.DataFrame(perm_rows,columns=["encoded_clinical_feature","accuracy_drop"]).sort_values("accuracy_drop",ascending=False)
display(perm_df.head(15))
fig,ax=plt.subplots(figsize=(9,5))
top=perm_df.head(15).sort_values("accuracy_drop")
ax.barh(top["encoded_clinical_feature"].astype(str),top["accuracy_drop"])
ax.set_title("Clinical encoded-feature permutation importance (demo)",fontweight="bold")
ax.set_xlabel("Accuracy drop after permutation"); plt.tight_layout(); plt.show()

# 3D input saliency for a single example, target = predicted class
demo_model.eval()
ct,mri,clin,label=next(iter(DataLoader(all_ds,batch_size=1,shuffle=False)))
ct=ct.to(DEVICE).requires_grad_(True); mri=mri.to(DEVICE).requires_grad_(True); clin=clin.to(DEVICE)
out=demo_model(ct,mri,clin)
target=int(out["logits"].argmax(1).item())
demo_model.zero_grad()
out["logits"][0,target].backward()
sal=ct.grad.detach().abs()[0,0].cpu().numpy()
mid=sal.shape[0]//2
fig,ax=plt.subplots(figsize=(6,5))
ax.imshow(ct.detach().cpu().numpy()[0,0,mid],cmap="gray")
ax.imshow(sal[mid],cmap="jet",alpha=.45)
ax.set_title(f"CT saliency overlay — predicted class {classes[target]}",fontweight="bold")
ax.axis("off"); plt.tight_layout(); plt.show()

## 11. Save model, metrics, predictions, Excel report, and ZIP bundle
For real data, also save the fitted preprocessing object and label map in a versioned experiment folder. The current notebook saves outputs to `OUTPUT_DIR`.

In [ ]:
OUTPUT_DIR = "/content/brain_tumor_multimodal_outputs" if os.path.isdir("/content") else str(Path.cwd()/"brain_tumor_multimodal_outputs")
os.makedirs(OUTPUT_DIR,exist_ok=True)

pred_df = pd.DataFrame({
    "patient_id":df["patient_id"].astype(str),
    "true_label":[classes[i] for i in y],
    "predicted_label":[classes[i] for i in oof_pred],
    "predicted_confidence":oof_probs.max(1),
    "uncertainty_proxy":oof_unc
})
for j,c in enumerate(classes): pred_df[f"probability_{c}"]=oof_probs[:,j]

metrics_df.to_csv(os.path.join(OUTPUT_DIR,"cross_validation_metrics.csv"),index=False)
ablation_df.to_csv(os.path.join(OUTPUT_DIR,"modality_ablation_demo.csv"),index=False)
pred_df.to_csv(os.path.join(OUTPUT_DIR,"out_of_fold_predictions.csv"),index=False)
perm_df.to_csv(os.path.join(OUTPUT_DIR,"clinical_permutation_importance_demo.csv"),index=False)
torch.save(demo_model.state_dict(),os.path.join(OUTPUT_DIR,"multimodal_brain_tumor_model.pth"))
with open(os.path.join(OUTPUT_DIR,"label_mapping.json"),"w") as f: json.dump({str(k):int(v) for k,v in label_map.items()},f,indent=2)

excel_path=os.path.join(OUTPUT_DIR,"brain_tumor_multimodal_report.xlsx")
with pd.ExcelWriter(excel_path,engine="openpyxl") as writer:
    metrics_df.to_excel(writer,sheet_name="CV Metrics",index=False)
    ablation_df.to_excel(writer,sheet_name="Modality Ablation Demo",index=False)
    pred_df.to_excel(writer,sheet_name="OOF Predictions",index=False)
    perm_df.to_excel(writer,sheet_name="Clinical Importance Demo",index=False)

zip_path=os.path.join(OUTPUT_DIR,"brain_tumor_multimodal_bundle.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for root,dirs,files in os.walk(OUTPUT_DIR):
        for file in files:
            full=os.path.join(root,file)
            if full != zip_path: z.write(full,arcname=os.path.relpath(full,OUTPUT_DIR))
print("Saved outputs to:",OUTPUT_DIR)
print("Excel:",excel_path)
print("ZIP:",zip_path)

## 12. Real-data checklist and limitations
1. Confirm patient IDs are unique and no patient appears in both training and validation.
2. Verify CT/MRI alignment and orientation, resample using physical spacing when metadata are available, and apply a validated brain-region extraction method.
3. N4 bias correction is MRI-specific and requires an appropriate medical-imaging toolkit (e.g., SimpleITK); this baseline does intensity normalization and resizing only.
4. The `StateSpaceInspiredMixer` is not official Mamba-2, and the 3D CNN is not a pretrained foundation model. For exact NeuroMamba-3D reproduction, obtain the intended architecture and pretrained checkpoint.
5. Use nested patient-level validation, calibration metrics (e.g., Brier score/ECE), confidence intervals, and an independent external test cohort before any clinical claim.
6. Ensure the target label definition is documented (e.g., tumor presence or tumor type). The uploaded methodology does not specify exact classes, dataset paths, or a public pretrained checkpoint.